# Project 4: Option portfolio under a spot shock

A synthetic interview practice case exploring pandas data preparation, matching market records to trades, and first-order option portfolio risk. This is learner-written work in progress, not a verified employer interview question.

## Problem and assumptions

A risk colleague wants to identify loss contributors under a hypothetical spot scenario and understand data coverage. Input files are in `data/04_option_portfolio_spot_shock/`.

Positive contract quantities are long; negative quantities are short. Delta is for a long option on one underlying unit, before contract size and position sign. Spot prices and scenario P&L are in euros. A shock of -5 means a 5% fall. Hold delta fixed and use:

`spot change = spot_eur * shock_pct / 100`

`scenario P&L = signed_contracts * units_per_contract * delta * spot change`

Keep one copy of identical market rows. Preserve trades with no matching market record as missing; unavailable P&L is not zero.

## Requested deliverables

1. Calculate P&L per trade and list trades without market data.
2. Report net P&L by underlying, the total for covered positions, and coverage as a count of original trades.
3. Show an independent hand calculation and one explicit duplicate or missing-match check.
4. Write a two- or three-sentence risk explanation naming the largest loss contributor, coverage and a limitation.

## Current status

The matching, trade-level calculations, grouped sums and covered-position total are implemented. **The coverage printout below is an unresolved draft error:** its expression counts excluded trades but labels them covered. Correcting the coverage report remains a learner task. The explicit missing-trade list, independent hand calculation, duplicate/missing-match demonstration and final risk explanation remain to be completed. The existing formula-agreement assertion checks consistency, not an independent benchmark.

## Load the synthetic data


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Run from the repository root or its notebooks directory.
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_dir = project_root / "data" / "04_option_portfolio_spot_shock"
if not data_dir.is_dir():
    raise FileNotFoundError("Open this notebook from the repository root or notebooks directory.")

positions = pd.read_csv(data_dir / "positions.csv")
market = pd.read_csv(data_dir / "market.csv").drop_duplicates()

print(market)
print(positions)


  underlying  spot_eur  shock_pct
0        ALP        80         -5
1        BRV       125         -4
3        CEN        40          5
  trade_id underlying  signed_contracts  units_per_contract  delta
0       T1        ALP                12                 100   0.60
1       T2        ALP                -5                 100   0.40
2       T3        BRV                 8                  50  -0.30
3       T4        CEN                -7                 100  -0.50
4       T5        DOR                 3                 100   0.55
5       T6        BRV                 5                 100   0.20


## Implementation and current checks

The original calculation logic is preserved, including the coverage reporting issue described above.


In [2]:
def scenario_PnL(signed_contracts, units_per_contract, delta, Delta_S):
    scenario_PnL = signed_contracts * units_per_contract * delta * Delta_S
    return scenario_PnL

# Calculating Delta_S using numpy arrays
spot = market["spot_eur"].to_numpy()
shock = market["shock_pct"].to_numpy()
Delta_S1 = np.array([spot[i] * (shock[i]/ 100) for i in range(len(spot))])

# Calculating Delta_S using panda DataFrames
def Delta_S2(spot, shock):
    Delta_S = np.array([])
    for i in range(len(market["spot_eur"])):
        Delta_S = np.append(Delta_S, spot.iloc[i] * (shock.iloc[i]/ 100))
    return Delta_S

# Merging the positions and market dataframes while matching the entries with the same "underlying" and retaining every left record, attaching matches
market["delta_S"] = Delta_S1
positions_new = positions.merge(
    market,
    how="left",
    on="underlying",
    validate="many_to_one"
)

# Calculating PnL and adding it to the positions dataframe
PnL = np.array([scenario_PnL(positions_new["signed_contracts"].iloc[i], positions_new["units_per_contract"].iloc[i], positions_new["delta"].iloc[i], positions_new["delta_S"].iloc[i]) for i in range(len(positions_new["trade_id"]))])

positions_new["PnL"] = PnL

# Group PnL for the same underlying
PnL_by_underlying = positions_new.groupby("underlying")["PnL"].sum(min_count=1).reset_index()

# Exclude entries with no market data
PnL_partial = PnL_by_underlying.dropna(subset=["PnL"])

# Checking that the scenario_PnL function works
PnL_check = []
for i in range(len(positions_new["trade_id"])):
    result = positions_new["signed_contracts"].iloc[i] * positions_new["units_per_contract"].iloc[i] * positions_new["delta"].iloc[i] * (positions_new["spot_eur"].iloc[i] *(positions_new["shock_pct"].iloc[i]/ 100))
    PnL_check.append(result)
assert np.allclose(PnL, PnL_check, equal_nan=True)



print(f"The number of original positions that were covered are {positions_new.shape[0] - positions_new.dropna(subset=["PnL"]).shape[0]}.")
print(f"The total PnL for covered positions is €{PnL_partial["PnL"].sum(min_count=1)}.")
print(PnL_by_underlying)
print(PnL_partial)

The number of original positions that were covered are 1.
The total PnL for covered positions is €-1280.0.
  underlying     PnL
0        ALP -2080.0
1        BRV   100.0
2        CEN   700.0
3        DOR     NaN
  underlying     PnL
0        ALP -2080.0
1        BRV   100.0
2        CEN   700.0


## Exploratory work

Retained experiments from learning Series, DataFrames and elementwise arithmetic.


In [3]:
#print(market["spot_eur"])
#print(market.iloc[:, 1])
#print(market["spot_eur"].to_numpy())
#print(market["spot_eur"].iloc[0])
#print(market[["spot_eur", "shock_pct"]].to_numpy())
#print(positions_new)
#print(positions_new.iloc[:, 2:7] + positions_new.iloc[:, 2:7])

def double(x):
    return 2 * x

#print(double(market["spot_eur"]))
#print(double(market["spot_eur"].to_numpy()))
#print(double(market[["spot_eur", "shock_pct"]]))
#print(double(market[["spot_eur", "shock_pct"]].to_numpy()))

print(positions_new)


print(PnL_check)

  trade_id underlying  signed_contracts  units_per_contract  delta  spot_eur  \
0       T1        ALP                12                 100   0.60      80.0   
1       T2        ALP                -5                 100   0.40      80.0   
2       T3        BRV                 8                  50  -0.30     125.0   
3       T4        CEN                -7                 100  -0.50      40.0   
4       T5        DOR                 3                 100   0.55       NaN   
5       T6        BRV                 5                 100   0.20     125.0   

   shock_pct  delta_S     PnL  
0       -5.0     -4.0 -2880.0  
1       -5.0     -4.0   800.0  
2       -4.0     -5.0   600.0  
3        5.0      2.0   700.0  
4        NaN      NaN     NaN  
5       -4.0     -5.0  -500.0  
[np.float64(-2880.0), np.float64(800.0), np.float64(600.0), np.float64(700.0), np.float64(nan), np.float64(-500.0)]
